# RAG Evaluation and Debugging
Companion notebook for the **RAG Evaluation and Debugging** section of Chapter 10.

Don't judge only the final answer -- measure **retrieval quality** and **answer quality** separately, because a bad answer can come from poor search, poor context selection, or poor prompting, and each needs a different fix.

The book's eval case and run result, as real Python objects:

In [1]:
eval_case = {
    "question": "What is the refund window for annual plans?",
    "expected_answer": "30 days",
    "expected_doc_ids": ["policy_2025_refunds"],
}

run_result = {
    "retrieved_doc_ids": ["policy_2025_refunds", "billing_faq"],
    "top_k_hit": True,
    "context_relevance": 0.84,
    "answer_relevance": 0.91,
    "faithfulness": 0.88,
}

**Hit rate**: how often the needed document actually appears in the top-k results, across a small eval set:

In [2]:
def hit_rate(cases: list[dict], results: list[dict]) -> float:
    hits = sum(
        1 for case, result in zip(cases, results)
        if set(case["expected_doc_ids"]) & set(result["retrieved_doc_ids"])
    )
    return hits / len(cases)


cases = [eval_case, {"question": "How do refunds work?", "expected_answer": "See policy", "expected_doc_ids": ["policy_2025_refunds"]}]
results = [run_result, {"retrieved_doc_ids": ["billing_faq"], "top_k_hit": False}]  # 2nd run missed the right doc
print("hit rate:", hit_rate(cases, results))

hit rate: 0.5


**Context relevance**, **answer relevance**, and **faithfulness** aren't given as code in the book -- here are small, clearly-labeled proxies (word-overlap based, not a real NLI model) that compute the same shape of score the book's `run_result` reports:

In [3]:
def context_relevance(question: str, retrieved_texts: list[str]) -> float:
    """Proxy: how much of the question's vocabulary shows up in the retrieved text."""
    q_words = {w.strip("?.,") for w in question.lower().split()}
    if not retrieved_texts or not q_words:
        return 0.0
    scores = [len(q_words & {w.strip("?.,") for w in t.lower().split()}) / len(q_words) for t in retrieved_texts]
    return sum(scores) / len(scores)


def answer_relevance(question: str, answer: str) -> float:
    """Proxy: does the answer's vocabulary actually engage with the question's?"""
    q_words = set(question.lower().split())
    a_words = set(answer.lower().split())
    return len(q_words & a_words) / max(len(q_words), 1)


def faithfulness(answer: str, retrieved_texts: list[str]) -> float:
    """Proxy: what fraction of the answer's words are actually supported by retrieved context.
    A low score is the tell for a hallucinated claim -- a number or fact the context never said."""
    context_words = {w.strip(".,") for w in " ".join(retrieved_texts).lower().split()}
    answer_words = [w.strip(".,") for w in answer.lower().split()]
    answer_words = [w for w in answer_words if w.isalpha() or w.isdigit()]
    if not answer_words:
        return 1.0
    supported = sum(1 for w in answer_words if w in context_words)
    return supported / len(answer_words)

In [4]:
context = ["Refunds are allowed within 30 days of purchase for annual plans."]
question = "What is the refund window for annual plans?"

grounded_answer = "The refund window for annual plans is 30 days."
hallucinated_answer = "The refund window for annual plans is 90 days, extendable on request."

print("context_relevance:", round(context_relevance(question, context), 2))
print("grounded answer   -> faithfulness:", round(faithfulness(grounded_answer, context), 2),
      " relevance:", round(answer_relevance(question, grounded_answer), 2))
print("hallucinated answer -> faithfulness:", round(faithfulness(hallucinated_answer, context), 2),
      " relevance:", round(answer_relevance(question, hallucinated_answer), 2))

context_relevance: 0.38
grounded answer   -> faithfulness: 0.56  relevance: 0.75
hallucinated answer -> faithfulness: 0.33  relevance: 0.75


The hallucinated answer's relevance looks fine (it still talks about refund windows), but its faithfulness drops -- "90" and "extendable" are never in the retrieved context. This is exactly why the book insists on measuring these separately: an answer can *look* relevant while being unsupported.

## A practical debugging loop
1. Collect failed queries. 2. Inspect missing or noisy chunks. 3. Check for unsupported claims. 4. Fix one layer at a time. 5. Rerun the same eval set.

In [5]:
def debug_failed_case(case: dict, result: dict, retrieved_texts: list[str], answer: str) -> dict:
    diagnosis = {"case": case["question"]}

    if not (set(case["expected_doc_ids"]) & set(result["retrieved_doc_ids"])):
        diagnosis["layer"] = "retrieval"
        diagnosis["reason"] = "expected document never retrieved -- check chunking, filters, or embedding mismatch"
    elif faithfulness(answer, retrieved_texts) < 0.45:
        diagnosis["layer"] = "generation"
        diagnosis["reason"] = "answer contains claims not supported by retrieved context"
    elif context_relevance(case["question"], retrieved_texts) < 0.3:
        diagnosis["layer"] = "context_selection"
        diagnosis["reason"] = "retrieved text is off-topic -- reranker or top_k likely needs tuning"
    else:
        diagnosis["layer"] = "none"
        diagnosis["reason"] = "no issue found"
    return diagnosis


failing_run = {"retrieved_doc_ids": ["policy_2025_refunds"]}
diagnosis = debug_failed_case(eval_case, failing_run, context, hallucinated_answer)
print("before fix:", diagnosis)

# Fix one layer at a time -- here, tighten generation to only restate retrieved numbers -- then rerun the same case.
diagnosis_after_fix = debug_failed_case(eval_case, failing_run, context, grounded_answer)
print("after fix: ", diagnosis_after_fix)

before fix: {'case': 'What is the refund window for annual plans?', 'layer': 'generation', 'reason': 'answer contains claims not supported by retrieved context'}
after fix:  {'case': 'What is the refund window for annual plans?', 'layer': 'none', 'reason': 'no issue found'}


Keep the eval set stable across reruns -- that's the only way a score change tells you the *system* changed, not the *test*.